In [3]:
import torch
import numpy as np

# Simple Attention Mechanism WithOut Trainable Weights

In [6]:
# for pytorch 
inputs = torch.tensor([
 [0.43, 0.15, 0.89], # Your (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts (x^3)
 [0.22, 0.58, 0.33], # with (x^4)
 [0.77, 0.25, 0.10], # one (x^5)
 [0.05, 0.80, 0.55] # step (x^6)
])
inputs.shape

torch.Size([6, 3])

In [ ]:
# e.g if I have "journey" word as out query then compute the attention scores with respect to each
query = inputs[1]

atten_scores = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    atten_scores[i] = torch.dot(x_i, query)
atten_scores


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [16]:
# Now convert these into attention weights 
atten_weights = atten_scores / atten_scores.sum()
print("Attention Weights --->  ", atten_weights)
print("Sum of Attention Weights ---> ", atten_weights.sum())

Attention Weights --->   tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of Attention Weights --->  tensor(1.0000)


In [20]:
# Compute the context vector 
ctx_vec = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    ctx_vec += atten_weights[i] * x_i
print(ctx_vec)


tensor([0.4355, 0.6451, 0.5680])


In [21]:
# In this example compute it respect to each
atten_scores_all = inputs @ inputs.T
atten_scores_all

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [25]:
atten_weights_all = torch.softmax(atten_scores_all, dim=-1)
atten_weights_all

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [27]:
ctx_vec_all = atten_weights_all @ inputs
ctx_vec_all

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
# for only query 1
class SimpleSelfAttentionV1():
    def __init__(self, inputs, query):
        self.inputs = inputs
        self.query = query

    def forward(self):
        # compute scores
        scores = self.inputs @ self.query

        # Compute attention weights 
        weights = F.softmax(scores, dim=0)

        # compute context vector 
        context = weights @ self.inputs

        return scores, weights, context

torch.manual_seed(123)
sv1 = SimpleSelfAttentionV1(inputs, query)
sv1.forward()


(tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]),
 tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]),
 tensor([0.4419, 0.6515, 0.5683]))

In [30]:
# for all
class SimpleSelfAttentionV2():
    def __init__(self, inputs):
        self.inputs = inputs

    def forward(self):
        # compute scores
        scores = self.inputs @ self.inputs.T

        # Compute attention weights 
        weights = F.softmax(scores, dim=0)

        # compute context vector 
        context = weights @ self.inputs

        return scores, weights, context

torch.manual_seed(123)
sv2 = SimpleSelfAttentionV2(inputs)
sv2.forward()


(tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
         [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
         [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
         [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
         [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
         [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]]),
 tensor([[0.2098, 0.1385, 0.1390, 0.1435, 0.1526, 0.1385],
         [0.2006, 0.2379, 0.2369, 0.2074, 0.1958, 0.2184],
         [0.1981, 0.2333, 0.2326, 0.2046, 0.1975, 0.2128],
         [0.1242, 0.1240, 0.1242, 0.1462, 0.1367, 0.1420],
         [0.1220, 0.1082, 0.1108, 0.1263, 0.1879, 0.0988],
         [0.1452, 0.1581, 0.1565, 0.1720, 0.1295, 0.1896]]),
 tensor([[0.4017, 0.5023, 0.5059],
         [0.5595, 0.7824, 0.6953],
         [0.5538, 0.7686, 0.6834],
         [0.3369, 0.4647, 0.4119],
         [0.3525, 0.4059, 0.3657],
         [0.3856, 0.5761, 0.5077]]))